# Retrieving additional metadata on movies using the API of TMDB

This script uses the API of The Movie Database in retrieving additional information on movies predefined in previous scripts. Additional information is collected on ratings and vote counts, ... . The API documentation can be consulted at; https://developer.themoviedb.org/reference/movie-details

In [2]:
import pandas as pd

#Read in the films from the existing dataset created in R
moviedata = pd.read_csv("../data/raw/raw_tmdb.csv")

print(moviedata.columns)

tmdb_id = moviedata['id'].unique().tolist()
print(f"\nAmount of movies in the dataset: {len(tmdb_id)}")

Index(['tconst', 'id', 'title_sanitycheck', 'TMDB_rating', 'TMDB_votecount',
       'timestamp1', 'budget', 'production_companies', 'production_countries',
       'timestamp2'],
      dtype='object')

Amount of movies in the dataset: 338


In [3]:
import requests
import json
import time

# API key
api_key = os.environ['TMDBkey']

extradata = []

for movie in tmdb_id:
    # API endpoint
    url = f"https://api.themoviedb.org/3/movie/{movie}/release_dates"
    
    # Parameters
    params = {
        "api_key": api_key
    }
    
    # Make the request
    response = requests.get(url, params=params)
    
    # Check if successful
    if response.status_code == 200:
        data = response.json()  # data IS the movie object directly

        # Confirm movie
        new_id = movie
        print(f"MOVIE FOUND with ID: {movie}")

        # Initialize release dates
        cinema_release = None
        digital_release = None
        
        # Retrieve release dates
        results = data.get("results", [])
        
        # Loop through countries to find US
        for country in results:
            if country["iso_3166_1"] == "US":
                # Loop through release dates for US
                release_dates = country.get("release_dates", [])
                
                for release in release_dates:
                    # Type 3 = Theatrical
                    if release["type"] == 3:
                        cinema_release = release["release_date"]
                    # Type 4 = Digital
                    elif release["type"] == 4:
                        digital_release = release["release_date"]
                
                break  # Stop after finding US
        
        # Confirm new data
        extradata.append({
            "id": new_id,
            "cinema_release": cinema_release,
            "digital_release": digital_release,
            "timestamp3": time.time()
        })
    else:
        print(f"Error {response.status_code}: Movie {movie} not found")
    
    time.sleep(.027)


# Convert new data to DataFrame
df_new = pd.DataFrame(extradata)

# Merge the two DataFrames on the id/tconst column
df_merged = pd.merge(moviedata, df_new, on='id', how='left')

# Save back to CSV (overwrite the original)
df_merged.to_csv('../data/raw/raw_tmdb.csv', index=False)

print(f"\nAdded new columns at {time.ctime(time.time())}!")

MOVIE FOUND with ID: 76600
MOVIE FOUND with ID: 83533
MOVIE FOUND with ID: 575265
MOVIE FOUND with ID: 385687
MOVIE FOUND with ID: 335977
MOVIE FOUND with ID: 575264
MOVIE FOUND with ID: 609681
MOVIE FOUND with ID: 447273
MOVIE FOUND with ID: 505642
MOVIE FOUND with ID: 447365
MOVIE FOUND with ID: 616037
MOVIE FOUND with ID: 447277
MOVIE FOUND with ID: 558449
MOVIE FOUND with ID: 1061474
MOVIE FOUND with ID: 1234821
MOVIE FOUND with ID: 572802
MOVIE FOUND with ID: 1022789
MOVIE FOUND with ID: 533535
MOVIE FOUND with ID: 453395
MOVIE FOUND with ID: 414906
MOVIE FOUND with ID: 617126
MOVIE FOUND with ID: 762509
MOVIE FOUND with ID: 640146
MOVIE FOUND with ID: 436270
MOVIE FOUND with ID: 976573
MOVIE FOUND with ID: 718789
MOVIE FOUND with ID: 298618
MOVIE FOUND with ID: 845781
MOVIE FOUND with ID: 338953
MOVIE FOUND with ID: 466420
MOVIE FOUND with ID: 1022796
MOVIE FOUND with ID: 753342
MOVIE FOUND with ID: 848538
MOVIE FOUND with ID: 667538
MOVIE FOUND with ID: 693134
MOVIE FOUND with I